# Project - Airline AI Assistant

 AI Customer Support assistant for an Airline

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Initialization
load_dotenv()

groq_api_key = os.getenv('GROQ_API_KEY')

groq_url = "https://api.groq.com/openai/v1"

groq = OpenAI(api_key=groq_api_key,base_url=groq_url)
# MODEL = "llama-3.1-8b-instant"
MODEL = "llama-3.3-70b-versatile" # Much better for logic and tool use

In [3]:

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.system_message =
"""

In [4]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    sql = conn.cursor()
    sql.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        sql = conn.cursor()
        sql.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = sql.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [5]:
get_ticket_price("Paris")

DATABASE TOOL CALLED: Getting price for Paris


'Ticket price to Paris is $899.0'

In [6]:
##to set ticket price
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        sql = conn.cursor()
        sql.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [ ]:
ticket_prices = {"london":699, "paris": 799, "tokyo": 1520, "sydney": 3999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [8]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [9]:
##without tools callback function for gradio
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

# gr.ChatInterface(fn=chat, type="messages").launch()

In [10]:
# with tools
# def chat(message, history):
#     history = [{"role":h["role"], "content":h["content"]} for h in history]
#     messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
#     response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)

#     while response.choices[0].finish_reason=="tool_calls":
#         message = response.choices[0].message
#         responses = handle_tool_calls(message)
#         messages.append(message)
#         messages.extend(responses)
#         response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
#     return response.choices[0].message.content

# with tools
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = ([{"role": "system", "content": system_message}]+ history + [{"role": "user", "content": message}])
    acknowledgements = [
        "ok", "okay", "thanks", "thank you",
        "cool", "great", "nice"
    ]

    # Disable tools for acknowledgement messages
    if message.lower().strip() in acknowledgements:
        response = groq.chat.completions.create(model=MODEL, messages=messages)
        return response.choices[0].message.content

    # Normal tool flow
    response = groq.chat.completions.create(model=MODEL,messages=messages,tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model=MODEL,messages=messages,tools=tools)
    return response.choices[0].message.content

In [11]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses


In [ ]:
view = gr.ChatInterface(fn=chat)
view.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for London
DATABASE TOOL CALLED: Getting price for Paris


In [18]:
view.close()

Closing server running on port: 7861


## 2. Text-to-Speech (GPT-4o-mini-tts Alternative)
For speech, Kokoro-82M is currently the best "small but mighty" open-source model. It is incredibly fast and sounds very natural. Another classic choice is pyttsx3 (completely offline, uses system voices) or coqui-tts.

Option A: Kokoro (High Quality)
Installation: pip install kokoro-onnx soundfile

In [19]:
import pyttsx3

def talker(message):
    engine = pyttsx3.init()
    engine.say(message)
    engine.runAndWait()

1. A multi-modal AI assistant with audio generation
2. Tool callling with database lookup
3. A step towards an Agentic workflow


In [35]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history     
    response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    # image = None 

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    # if cities:
    #     image = artist(cities[0])
    
    # return history, voice, image
    return history, voice

In [21]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

In [36]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
        # image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output]
    )

# ui.launch(inbrowser=True, auth=("ned", "bananas"))
ui.launch(auth=("ned", "bananas"))

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [37]:
ui.close()

Closing server running on port: 7867
